# University of London - ML Code - Computer Science Final Project

**BSc Computer Science**

**Subject: CM3070 Computer Science Final Project**

**Student: In Final Project report**

**Student Number: In Final Project report**

## ML Process

Based on the knowledge gathered from the experiments, this notebook aims to create 3 models.

## Preparing the data

### Transforming the csv data to a numpy array

In [1]:
import numpy as np

str_to_np_date = lambda x: np.datetime64(x)

usdYen_raw_data = np.genfromtxt("./data/currency-data/USD-JPY-DAILY.csv", skip_header=1, delimiter=";", usecols=1)
usdYen_raw_data_dates = np.genfromtxt("./data/currency-data/USD-JPY-DAILY.csv", skip_header=1, delimiter=";", usecols=0, converters={0: str_to_np_date})

print("Length: ",len(usdYen_raw_data))
print("Data type: ",usdYen_raw_data.dtype)
print("Raw Data: ",usdYen_raw_data)
print("Raw Data Dates: ",usdYen_raw_data_dates)

Length:  5000
Data type:  float64
Raw Data:  [154.71 155.21 155.81 ... 118.22 118.89 118.46]
Raw Data Dates:  ['2025-12-16' '2025-12-15' '2025-12-12' ... '2006-10-19' '2006-10-18'
 '2006-10-17']


As the currency data is from newer to older, the order should be inverted.

In [2]:
usdYen_raw_data = np.flip(usdYen_raw_data, axis=0)
print(usdYen_raw_data)

[118.46 118.89 118.22 ... 155.81 155.21 154.71]


### Computing the numer of samples for each data split

In [3]:
train_samples_number = len(usdYen_raw_data)
print("Number of train samples: ", train_samples_number)

Number of train samples:  5000


### Creating timeseries data

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [4]:
from tensorflow import keras

# Parameters
sampling_rate = 1
sequence_length = 150 # Observations will go back 150 days
delay = sampling_rate * (sequence_length + 5 - 1) # target is 5 days after the end of the sequence
batch_size = train_samples_number

# train dataset
train_dataset = keras.utils.timeseries_dataset_from_array(
    usdYen_raw_data,
    targets=usdYen_raw_data[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    batch_size=batch_size,
)

### - Checking that timeseries data works correctly

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [5]:
'''
for inputs, targets in train_dataset:
    for i in range(inputs.shape[0]):
        print([float(x) for x in inputs[i]], float(targets[i]))
'''

'\nfor inputs, targets in train_dataset:\n    for i in range(inputs.shape[0]):\n        print([float(x) for x in inputs[i]], float(targets[i]))\n'

### - Extracting data inputs and outputs

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [6]:
import tensorflow as tf

data_inputs = []
data_outputs = []

for samples, targets in train_dataset:
    print("Samples: ", samples)
    print("Sample shape: ", samples.shape)
    print("Targets: ", targets)
    print("Targets shape: ", targets.shape)
    data_inputs = tf.make_ndarray(tf.make_tensor_proto(samples))
    data_outputs = tf.reshape(tf.make_ndarray(tf.make_tensor_proto(targets)), [-1,1])

data_inputs_test = data_inputs[-200:]
data_outputs_test = data_outputs[-200:]
data_inputs = data_inputs[:-200]
data_outputs = data_outputs[:-200]

print("Data Inputs: ", len(data_inputs))
print("Data Inputs Test: ", len(data_inputs_test))
print("Data Outputs: ", len(data_outputs))
print("Data Outputs Test: ", len(data_outputs_test))
    

Samples:  tf.Tensor(
[[118.46  118.89  118.22  ... 119.88  120.18  120.34 ]
 [118.89  118.22  118.7   ... 120.18  120.34  120.26 ]
 [118.22  118.7   119.34  ... 120.34  120.26  120.8  ]
 ...
 [148.35  147.453 146.63  ... 155.24  155.08  155.34 ]
 [147.453 146.63  145.592 ... 155.08  155.34  155.92 ]
 [146.63  145.592 145.581 ... 155.34  155.92  156.86 ]], shape=(4846, 150), dtype=float64)
Sample shape:  (4846, 150)
Targets:  tf.Tensor([121.45 121.54 121.65 ... 155.81 155.21 154.71], shape=(4846,), dtype=float64)
Targets shape:  (4846,)
Data Inputs:  4646
Data Inputs Test:  200
Data Outputs:  4646
Data Outputs Test:  200


2026-01-21 15:13:46.419253: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [7]:
print("----")
print("Input Data: ", data_inputs)
print("----")
print("Output Data: ", data_outputs)
print("----")

----
Input Data:  [[118.46  118.89  118.22  ... 119.88  120.18  120.34 ]
 [118.89  118.22  118.7   ... 120.18  120.34  120.26 ]
 [118.22  118.7   119.34  ... 120.34  120.26  120.8  ]
 ...
 [144.052 144.518 146.702 ... 149.076 149.801 150.545]
 [144.518 146.702 147.181 ... 149.801 150.545 149.485]
 [146.702 147.181 146.634 ... 150.545 149.485 149.801]]
----
Output Data:  tf.Tensor(
[[121.45 ]
 [121.54 ]
 [121.65 ]
 ...
 [148.018]
 [147.151]
 [147.708]], shape=(4646, 1), dtype=float64)
----


## Baseline: Basic machine learning model

The baseline will be based on the code provided in Chapter 10.2.3 (Deep Learning For Timeseries, Let’s try a basic machine learning model) of the book Deep Learning with Python by Francois Chollet [2].

In [8]:
from keras import models
from keras import layers
from keras import activations

def build_baseline_basic_model():
    model = models.Sequential()
    model.add(layers.Dense(16, input_shape=(sequence_length, 1),activation=activations.relu))
    model.add(layers.Dense(1, activation=activations.linear))
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    model.summary()
    return model

baseline_basic_model = build_baseline_basic_model()

/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 150, 16)        │            32 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 150, 1)         │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49 (196.00 B)

 Trainable params: 49 (196.00 B)

 Non-trainable params: 0 (0.00 B)

Keras checkpoint to save model with best performance

In [9]:
baseline_basic_model_callbacks = [
    keras.callbacks.ModelCheckpoint("models/baseline_basic_model.keras", save_best_only=True)
]

Evalute it on the train dataset

In [10]:
history_baseline_basic_model = baseline_basic_model.fit(
    data_inputs,
    data_outputs,
    epochs=10,
    validation_data=(data_inputs_test, data_outputs_test),
    callbacks=baseline_basic_model_callbacks
)

Epoch 1/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 11079.1602 - mae: 101.3782 - val_loss: 7737.9175 - val_mae: 87.8349
Epoch 2/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 691us/step - loss: 1418.4597 - mae: 31.3413 - val_loss: 45.6444 - val_mae: 5.6877
Epoch 3/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 658us/step - loss: 40.7284 - mae: 4.6139 - val_loss: 45.6717 - val_mae: 5.5205
Epoch 4/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 678us/step - loss: 40.7536 - mae: 4.6291 - val_loss: 44.0196 - val_mae: 5.4977
Epoch 5/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 644us/step - loss: 40.7669 - mae: 4.6201 - val_loss: 57.2601 - val_mae: 6.0189
Epoch 6/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 656us/step - loss: 40.7511 - mae: 4.6243 - val_loss: 50.8502 - val_mae: 5.7263
Epoch 7/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 644us/step - loss: 40.6554 - mae: 4.6219 - val_loss: 53.4156 - val_mae: 5.8399
Epoch 8/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 680us/step - loss: 40.7818 - mae: 4.6216 - val_loss: 44.5954 - val_mae: 5.4917
Epoch 9/10
146/

Results

In [14]:
print(f"Train MAE: {min(history_baseline_basic_model.history['mae'])}")
print(f"Test MAE: {min(history_baseline_basic_model.history['val_mae'])}")

Train MAE: 4.61392879486084
Test MAE: 5.491413116455078


## Convolutional model

The baseline will be based on the code provided in Chapter 10.2.4 (Deep Learning For Timeseries, Let’s try a 1D convolutional model) of the book Deep Learning with Python by Francois Chollet [3].

In [15]:
def build_convolutional_model():
    model = models.Sequential()
    model.add(layers.Conv1D(8, 24, input_shape=(sequence_length, 1), activation=activations.relu))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Conv1D(8, 12, activation=activations.relu))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Conv1D(8, 6, activation=activations.relu))
    model.add(layers.GlobalAveragePooling1D())
    model.add(layers.Dense(1, activation=activations.linear))
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    model.summary()
    return model

convolutional_model = build_convolutional_model()

/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 127, 8)         │           200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 63, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 52, 8)          │           776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 26, 8)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 21, 8)          │           392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 8)              │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,377 (5.38 KB)

 Trainable params: 1,377 (5.38 KB)

 Non-trainable params: 0 (0.00 B)

Keras checkpoint to save model with best performance

In [16]:
convolutional_model_callbacks = [
    keras.callbacks.ModelCheckpoint("models/convolutional_model.keras", save_best_only=True)
]

Evalute it on the train dataset

In [17]:
history_convolutional_model = convolutional_model.fit(
    data_inputs,
    data_outputs,
    epochs=10,
    validation_data=(data_inputs_test, data_outputs_test),
    callbacks=convolutional_model_callbacks
)

Epoch 1/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 172.2192 - mae: 7.7900 - val_loss: 42.6407 - val_mae: 5.7105
Epoch 2/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 52.7513 - mae: 5.6833 - val_loss: 109.5295 - val_mae: 8.4864
Epoch 3/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 49.2817 - mae: 5.5206 - val_loss: 92.7033 - val_mae: 7.9268
Epoch 4/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 48.7339 - mae: 5.4133 - val_loss: 377.0358 - val_mae: 18.2313
Epoch 5/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 50.6842 - mae: 5.5338 - val_loss: 68.5703 - val_mae: 6.8248
Epoch 6/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 49.1238 - mae: 5.4323 - val_loss: 180.5171 - val_mae: 11.7399
Epoch 7/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 48.8811 - mae: 5.4280 - val_loss: 176.0932 - val_mae: 11.5597
Epoch 8/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 47.0579 - mae: 5.3089 - val_loss: 46.2099 - val_mae: 5.5413
Epoch 9/10
146/146 ━━━━━━━━━━━━━

As expected, the model performed worse than the baseline one (Dense model).

## HyperTuner

The code is based on the code provided in the Keras documentation for the KerasTuner [4].

In [ ]:
from keras import models
from keras import layers

def build_lstm_model(hp):
    model = models.Sequential()
    model.add(
        layers.LSTM(
            units = hp.Int("units", min_value=16, max_value=64, step=4),
            input_shape=(sequence_length, 1)
        )
    )
    
    for i in range(2):
        model.add(
            layers.Dense(
                units = hp.Int(f"units_{i}", min_value=4, max_value=56, step=4)
            )
        )
    
    model.add(
        layers.Dense(
            units = 1
        )
    )
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

In [ ]:
import keras_tuner

tuner = keras_tuner.RandomSearch(
    hypermodel=build_lstm_model,
    objective="val_mae",
    max_trials=10,
    executions_per_trial=4,
    overwrite=True,
    directory="tuner_results/final_project_tuner_experimentNineteen_results",
    project_name="final_project_tunerfinal_project_tuner_experimentNineteen"
)

tuner.search_space_summary()

In [ ]:
tuner.search(data_inputs, data_outputs, epochs=5, validation_data=(data_inputs_test, data_outputs_test))

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

## Genetic Algorithm

The code for the Genetic Algorithm is based on the code provided by the PyGAD library documentation [5]. But for this experiment this is not necessary.

## About the code

The Genetic Algorithm code is based on the code shown in the docs of the PyGAD library.

The timeseries data code is based on the code shown in the chapter of the book Deep Learning with Python.

## References

1- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries, Preparing the data. Retrieved from https://learning.oreilly.com/library/view/deep-learning-with/9781617296864/Text/10.htm#heading_id_5

2- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries, Let’s try a basic machine learning model. Retrieved from https://learning.oreilly.com/library/view/deep-learning-with/9781617296864/Text/10.htm#heading_id_7

3- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries, Let’s try a 1D convolutional model. Retrieved from https://learning.oreilly.com/library/view/deep-learning-with/9781617296864/Text/10.htm#heading_id_8

4- Keras. Getting started with KerasTuner. Retrieved from https://keras.io/keras_tuner/getting_started/

5- PyGAD. pygad.kerasga Module. Retrieved from https://pygad.readthedocs.io/en/latest/kerasga.html#